# 进阶教程（十一）：部署与可观测

> 学会的链要能上线服务，上线的服务要能看见它在干什么。
> 本讲用 FastAPI 把链部署成 REST API（含流式 SSE），并用 LangSmith 做可观测。

## 本讲内容
1. 部署方式选型：LangServe vs FastAPI 手写 vs LangGraph Platform
2. FastAPI + SSE 流式端点（实测可用，langserve 未装时的手写方案）
3. 用 TestClient 离线测试端点
4. LangSmith 追踪与本地降级

# 0. 环境准备与运行说明

**前置要求：**
- 根目录 `.env` 已配置 `DEEPSEEK_API_KEY`（本教程用真实 DeepSeek，无本地降级）
- 已安装：`langchain>=1.3`、`langgraph>=1.2`、`langchain-classic`、`langchain-deepseek`
- 检索章节使用本地 `BAAI/bge-small-zh-v1.5` 嵌入（首次运行会下载模型，约 100MB）

**运行说明：**
- 按 cell 顺序执行；除标注外，每个示例消耗少量 API 额度（单次 < 0.01 元量级）
- 本教程面向已学完 `advanced_tutorial/01-06` 的开发者
- 各讲结尾 FAQ 汇总了基于 langchain 1.3.14 实测的导入路径与坑

In [ ]:

# ========== 0. 初始化（每个 notebook 第一格） ==========
import os, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=DeprecationWarning)

# HF 镜像必须先于任何 langchain/huggingface 导入设置
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")

ROOT = Path.cwd().parent  # advanced_tutorial 的上一级 = 项目根目录
sys.path.insert(0, str(ROOT))
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

assert os.getenv("DEEPSEEK_API_KEY"), "请先在根目录 .env 配置 DEEPSEEK_API_KEY"

from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(model="deepseek-chat", temperature=0.2)
print("模型就绪:", model.__class__.__name__)


## 1. 部署方式选型

| 方案 | 说明 | 状态 |
|---|---|---|
| **LangServe** | 官方"链→REST"框架，`add_routes(app, chain)` 一条命令 | 已进入维护模式，官方重心转向 LangGraph Platform |
| **FastAPI 手写** | 自己写端点，`astream` + SSE | 本项目采用（langserve 未安装，且更可控） |
| **LangGraph Platform** | 官方托管部署（云/自托管） | 生产大规模方案，本教程不展开 |

> 本讲用 **FastAPI 手写**：你已经会 SSE（langchain_tutorial/11），会 LCEL，
> 两者拼起来就是部署。

## 2. FastAPI + SSE 流式端点

把一条 RAG 链包成两个端点：`/ask`（非流式）和 `/ask/stream`（SSE 流式）：

In [ ]:

from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import json

# 一条极简链（生产换成你的 RAG 链）
chain = ChatPromptTemplate.from_template("用一句话回答：{q}") | model | StrOutputParser()

app = FastAPI(title="我的 RAG 服务")

@app.get("/ask")
def ask(q: str):
    """非流式：直接返回完整回答"""
    return {"answer": chain.invoke({"q": q})}

@app.get("/ask/stream")
def ask_stream(q: str):
    """流式：SSE 推送，逐 token 返回"""
    def gen():
        for token in chain.stream({"q": q}):
            yield f"data: {json.dumps({'t': token}, ensure_ascii=False)}\n\n"
        yield "data: [DONE]\n\n"
    return StreamingResponse(gen(), media_type="text/event-stream")

print("FastAPI 服务已定义:", app.title)

**要点**：SSE 格式为 `data: <json>\n\n`，用 `[DONE]` 标记结束。
前端用 `EventSource` 即可消费，与 langchain_tutorial/11 讲完全一致。

## 3. 用 TestClient 离线测试（不起真实服务）

notebook 里不适合起常驻进程，用 FastAPI 的 TestClient 直接测端点：

In [ ]:

from fastapi.testclient import TestClient

client = TestClient(app)

# 非流式端点
r = client.get("/ask", params={"q": "什么是向量检索"})
print("非流式:", r.json()["answer"][:60])

# 流式端点：逐行读取 SSE
print("流式: ", end="")
with client.stream("GET", "/ask/stream", params={"q": "什么是 Agent"}) as resp:
    for line in resp.iter_lines():
        if line.startswith("data:") and "[DONE]" not in line:
            token = json.loads(line[5:])["t"]
            print(token, end="", flush=True)
print()

## 4. LangSmith 可观测

部署后需要"看见"每次调用：token、耗时、调用树。
LangSmith 是官方观测平台，根 `.env` 已配好 `LANGSMITH_*`，**零代码接入**：

In [ ]:

import os
print("LANGSMITH_TRACING:", os.getenv("LANGSMITH_TRACING"))
print("LANGSMITH_PROJECT:", os.getenv("LANGSMITH_PROJECT_NAME"))
print("LANGSMITH_API_KEY:", "已配置" if os.getenv("LANGSMITH_API_KEY") else "未配置")

**LangSmith 能看到**：完整调用树、每步输入输出、token/费用、延迟瀑布图。

**无 LangSmith key 的本地替代**：用 `astream_events` 自建日志（04 讲的自定义回调 Meter 也是同理）：

In [ ]:

import time

# 极简本地观测：统计调用与耗时（不依赖任何外部服务）
class LocalTrace:
    def __init__(self):
        self.t0 = time.time()
        self.events = []
    def log(self, e):
        self.events.append((round(time.time() - self.t0, 2), e))

trace = LocalTrace()
trace.log("start")
r = model.invoke("用 20 字说明什么是可观测性")
trace.log("llm_done")
for t, e in trace.events:
    print(f"  +{t}s  {e}")
print("回答:", r.content)

**要点**：LangSmith 适合生产（有 UI、有聚合），本地 `astream_events`/回调适合
调试（零依赖）。两者并不互斥——LangSmith 本身就是基于回调体系实现的。

## 5. 常见问题（FAQ）

| 问题 | 原因 | 解决 |
|---|---|---|
| `langserve` 导入失败 | 未安装/已维护模式 | 用 FastAPI + SSE 手写（本讲方案） |
| SSE 前端收不到数据 | 缺 `media_type="text/event-stream"` | 显式设置 media_type |
| notebook 里起 uvicorn 卡死 | 常驻进程阻塞 cell | 用 TestClient，或子进程 + 手动关闭 |
| LangSmith 看不到数据 | tracing 未开/网络不通 | 检查 .env 的 LANGSMITH_TRACING 与网络 |
| TestClient 报 deprecation | starlette 提示换 httpx2 | 忽略即可，接口仍可用 |